# 06 • Register overlapping acquisitions and audit transcript duplication

**New upstream pilot stage: do not run segmentation until registration has been reviewed.**

This notebook reads the exact selected windows from notebook 00 and expands to nearby source
FOVs for registration context. It keeps your reviewed image/label flips and does not edit any
source Zarr. `rgb_v2_legacy_thresholds` is frozen for the later before/after experiment.

The first implementation uses the installed SciPy/scikit-image stack: normalized template
correlation on multiple independent border strips, followed by a translation-graph solve.
It does not require HALO or a package upgrade. `multiview-stitcher` remains a good candidate
for scaling the registration backend after this selected-field test.

**Not a whole-sample stitch:** disjoint selected neighborhoods are anchored independently to
one original FOV. Do not concatenate their cell outputs into a single biological sample yet.

References: [scikit-image template correlation](https://scikit-image.org/docs/0.26.x/api/skimage.feature.html#skimage.feature.match_template),
[SciPy inverse image resampling](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.affine_transform.html),
[multiview-stitcher registration design](https://multiview-stitcher.github.io/multiview-stitcher/main/registration_overview/).

In [1]:
from pathlib import Path
import sys, importlib, json
import numpy as np
import pandas as pd

HERE = Path.cwd()  # Set to the existing pilot directory if necessary.
for name in ('cosmx_rgb_pilot.py', 'cosmx_pilot_review.py', 'cosmx_overlap_stitch.py', 'pilot_config.json'):
    if not (HERE / name).is_file():
        raise FileNotFoundError(f"Keep the add-on beside the existing pilot bundle. Missing {HERE / name}")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import cosmx_rgb_pilot as pilot
import cosmx_pilot_review as review
import cosmx_overlap_stitch as stitch
importlib.reload(pilot)
importlib.reload(review)
importlib.reload(stitch)

# Your edited baseline configuration is read, never overwritten.
config = pilot.read_json(HERE / 'pilot_config.json')
BASELINE_ROOT = Path(config['output_root'])
WORKSPACE = BASELINE_ROOT.with_name(BASELINE_ROOT.name + '_registered_owner_v1')
SAMPLE_KEYS = list(config['samples'])  # Both colon samples by default.
print('Baseline:', BASELINE_ROOT)
print('New registered workspace:', WORKSPACE)
print('Add-on version:', stitch.VERSION)
print('Samples:', SAMPLE_KEYS)

Baseline: /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/cpsam_v2_rgb_softstitch_pilot_v1
New registered workspace: /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/cpsam_v2_rgb_softstitch_pilot_v1_registered_owner_v1
Add-on version: 1.0.0
Samples: ['B20317610105_colon', 'B20971880114_colon']


## 1. Inventory the selected fields and freeze a registration recipe

Only source FOVs around these windows are used. The routine tests image content; it does not
assume that metadata saying “abut” means there is no physical overlap. Default search parameters
cover roughly **28–280 px of normal overlap** and **±64 px tangential displacement**. They are
search bounds, not assertions that every seam has 50 or 100 px overlap.

The blue composite channel is used for registration; final segmentation still uses all RGB
channels. `green`, `red`, and `rgb_mean` can be tested in a **new workspace revision** when the
blue channel has insufficient structure. Do not lower quality gates merely to force acceptance.

In [2]:
REGISTRATION_OPTIONS = dict(stitch.DEFAULT_REGISTRATION)
# Examples of a deliberate new experiment:
# REGISTRATION_OPTIONS['channel'] = 'rgb_mean'
# WORKSPACE = BASELINE_ROOT.with_name(BASELINE_ROOT.name + '_registered_owner_rgbmean_v2')

scope_rows = []
for sample in SAMPLE_KEYS:
    layout, selected_tiles, input_hashes = stitch.selected_plan(config, sample)
    scope = stitch.registration_scope(layout, selected_tiles, REGISTRATION_OPTIONS)
    pairs = stitch.candidate_pairs(scope,
        tolerance_px=REGISTRATION_OPTIONS['candidate_edge_tolerance_px'],
        min_shared_px=REGISTRATION_OPTIONS['min_shared_edge_px'])
    for tile in selected_tiles:
        scope_rows.append({'sample': sample, 'tile': tile['tile_id'],
                           'reason': tile.get('reason', ''),
                           'original_FOVs_in_context': len(scope),
                           'candidate_edges_in_context': len(pairs)})
display(pd.DataFrame(scope_rows))
display(pd.Series(REGISTRATION_OPTIONS, name='registration_recipe'))

,sample,tile,reason,original_FOVs_in_context,candidate_edges_in_context
0,B20317610105_colon,pilot_00_vertical_seam,vertical_seam,18,23
1,B20317610105_colon,pilot_01_horizontal_seam,horizontal_seam,18,23
2,B20317610105_colon,pilot_02_four_fov_junction,four_fov_junction,18,23
3,B20317610105_colon,pilot_03_dense_interior,dense_interior,18,23
4,B20971880114_colon,pilot_00_vertical_seam,vertical_seam,25,30
5,B20971880114_colon,pilot_01_horizontal_seam,horizontal_seam,25,30
6,B20971880114_colon,pilot_02_four_fov_junction,four_fov_junction,25,30
7,B20971880114_colon,pilot_03_dense_interior,dense_interior,25,30


channel                        blue
search_depth_px                 256
template_width_px                24
edge_inset_px                     4
tangent_length_px               384
tangent_search_px                64
n_strips                          5
candidate_edge_tolerance_px     320
min_shared_edge_px              512
highpass_sigma_px               8.0
min_ncc                        0.45
min_peak_margin                0.03
peak_exclusion_px                 6
max_strip_deviation_px          3.0
max_holdout_error_px            3.0
max_graph_residual_px           3.0
context_margin_px               512
max_component_shift_px         1536
Name: registration_recipe, dtype: object

## 2. Fit image-based translations; save strip and graph diagnostics

Five nonoverlapping strips are sampled along each source edge. Alternating strips estimate the
translation; the other strips validate it. Low-texture, ambiguous, search-boundary, or inconsistent
matches are rejected. A graph solve checks consistency where loops exist. A tree has no loop
constraint, so visual inspection remains necessary.

`candidate_pass` is **not approval**. Templates with repeated crypt/cell patterns can create false
matches even with high correlation. Inspect the registered source-A/source-B landmarks below.

Runs resume by request hash. A changed source, selected tile, orientation, recipe, or manual
correction requires a new workspace; no stale result is silently reused.

In [3]:
RUN_REGISTRATION = True
MANUAL_EDGE_CORRECTIONS = {}  # Optional, sample -> pair_id -> explicitly reviewed dx/dy/reason.
# Example format only; never insert an unverified shift:
# {'B20317610105_colon': {'12__13__vertical':
#     {'dx_px': -50.0, 'dy_px': 0.0, 'reviewed': True, 'reason': 'Landmarks manually verified'}}}

candidates = {}
for sample in SAMPLE_KEYS:
    if RUN_REGISTRATION:
        candidate, pair_qc, strip_qc = stitch.register_selected_sample(
            config, sample, WORKSPACE, options=REGISTRATION_OPTIONS,
            manual=MANUAL_EDGE_CORRECTIONS.get(sample))
    else:
        folder = WORKSPACE / sample / 'registration'
        candidate = pilot.read_json(folder / 'candidate.json')
        pair_qc = pd.read_csv(folder / 'pair_summary.csv')
    candidates[sample] = candidate
    print(sample, candidate['registration_status'])
    if candidate.get('blocking_error'):
        print(candidate['blocking_error'])
    display(pair_qc)
    if candidate['registration_status'] == 'candidate_ready':
        display(pd.read_csv(WORKSPACE / sample / 'registration' / 'graph_residuals.csv'))
        print('Solution hash:', candidate['solution_hash'])

[registration 1/23] B20317610105_colon 87__88__vertical


/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve

[registration 2/23] B20317610105_colon 99__87__horizontal
[registration 3/23] B20317610105_colon 88__89__vertical
[registration 4/23] B20317610105_colon 100__88__horizontal
[registration 5/23] B20317610105_colon 101__89__horizontal
[registration 6/23] B20317610105_colon 94__95__vertical
[registration 7/23] B20317610105_colon 106__94__horizontal
[registration 8/23] B20317610105_colon 95__96__vertical
[registration 9/23] B20317610105_colon 107__95__horizontal
[registration 10/23] B20317610105_colon 108__96__horizontal
[registration 11/23] B20317610105_colon 99__100__vertical
[registration 12/23] B20317610105_colon 111__99__horizontal
[registration 13/23] B20317610105_colon 100__101__vertical
[registration 14/23] B20317610105_colon 112__100__horizontal
[registration 15/23] B20317610105_colon 113__101__horizontal
[registration 16/23] B20317610105_colon 106__107__vertical
[registration 17/23] B20317610105_colon 107__108__vertical
[registration 18/23] B20317610105_colon 110__111__vertical
[r

,pair_id,a,b,axis,nominal_gap_px,tangent0,tangent1,status,dx_px,dy_px,n_fit_good,n_holdout_good,fit_max_deviation_px,holdout_max_error_px,median_ncc,reason
0,87__88__vertical,87,88,vertical,0.0,59584.0,63840.0,candidate_pass,-50.167662,-8.967595,2,2,1.754043,1.656173,0.841386,fit and held-out strips agree
1,99__87__horizontal,99,87,horizontal,0.0,68096.0,72352.0,needs_review,9.272639,-48.811347,3,2,4.603935,4.692982,0.833332,strip shifts disagree
2,88__89__vertical,88,89,vertical,0.0,59584.0,63840.0,candidate_pass,-52.300299,-8.069557,2,1,1.991200,1.711192,0.838733,fit and held-out strips agree
3,100__88__horizontal,100,88,horizontal,0.0,72352.0,76608.0,needs_review,9.394796,-49.487675,3,2,4.311070,3.710744,0.915060,strip shifts disagree
4,101__89__horizontal,101,89,horizontal,0.0,76608.0,80864.0,needs_review,9.245901,-50.013414,3,2,4.363698,2.913475,0.859221,strip shifts disagree
5,94__95__vertical,94,95,vertical,0.0,55328.0,59584.0,needs_review,-44.674225,-9.307142,3,2,3.714558,3.461256,0.884100,strip shifts disagree
6,106__94__horizontal,106,94,horizontal,0.0,46816.0,51072.0,needs_review,9.053467,-49.465648,3,2,4.574442,3.944611,0.902285,strip shifts disagree
7,95__96__vertical,95,96,vertical,0.0,55328.0,59584.0,needs_review,-49.052841,-9.828856,3,2,3.525360,2.297000,0.897374,strip shifts disagree
8,107__95__horizontal,107,95,horizontal,0.0,51072.0,55328.0,needs_review,8.917737,-49.092932,3,2,4.441138,4.046201,0.899424,strip shifts disagree
9,108__96__horizontal,108,96,horizontal,0.0,55328.0,59584.0,needs_review,8.984808,-49.659706,3,2,3.948410,3.488802,0.829890,strip shifts disagree


[registration 1/30] B20971880114_colon 7__8__vertical


/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/code/Python/cosmx_cpsamv2_softstitch_pilot/cosmx_rgb_pilot.py:88: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  return sd.read_zarr(str(path))
no parent found for <ome_zarr.reader.Label object at 0x7f41fc392bd0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f9de50>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f7e570>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f055b0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f062d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f07230>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f07ef0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f07950>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f05d30>: 

[registration 2/30] B20971880114_colon 18__7__horizontal
[registration 3/30] B20971880114_colon 8__9__vertical
[registration 4/30] B20971880114_colon 19__8__horizontal
[registration 5/30] B20971880114_colon 20__9__horizontal
[registration 6/30] B20971880114_colon 18__19__vertical
[registration 7/30] B20971880114_colon 29__18__horizontal
[registration 8/30] B20971880114_colon 19__20__vertical
[registration 9/30] B20971880114_colon 30__19__horizontal
[registration 10/30] B20971880114_colon 31__20__horizontal
[registration 11/30] B20971880114_colon 29__30__vertical
[registration 12/30] B20971880114_colon 30__31__vertical
[registration 13/30] B20971880114_colon 60__61__vertical
[registration 14/30] B20971880114_colon 71__60__horizontal
[registration 15/30] B20971880114_colon 72__61__horizontal
[registration 16/30] B20971880114_colon 71__72__vertical
[registration 17/30] B20971880114_colon 91__92__vertical
[registration 18/30] B20971880114_colon 101__91__horizontal
[registration 19/30] B209

,pair_id,a,b,axis,nominal_gap_px,tangent0,tangent1,status,dx_px,dy_px,n_fit_good,n_holdout_good,fit_max_deviation_px,holdout_max_error_px,median_ncc,reason
0,7__8__vertical,7,8,vertical,0.0,63840.0,68096.0,needs_review,-49.331744,-9.125148,3,2,4.627823,4.653650,0.749117,strip shifts disagree
1,18__7__horizontal,18,7,horizontal,0.0,42560.0,46816.0,needs_review,9.128736,-48.632704,3,2,4.548105,4.087785,0.835516,strip shifts disagree
2,8__9__vertical,8,9,vertical,0.0,63840.0,68096.0,needs_review,-48.836468,-9.403120,3,2,3.592992,3.510895,0.768509,strip shifts disagree
3,19__8__horizontal,19,8,horizontal,0.0,46816.0,51072.0,needs_review,9.298668,-48.660307,3,1,5.067438,3.849672,0.794279,strip shifts disagree
4,20__9__horizontal,20,9,horizontal,0.0,51072.0,55328.0,candidate_pass,8.250033,-51.060083,2,2,2.430227,2.562450,0.724481,fit and held-out strips agree
5,18__19__vertical,18,19,vertical,0.0,59584.0,63840.0,needs_review,-48.680459,-9.317891,3,2,3.890922,4.094318,0.737049,strip shifts disagree
6,29__18__horizontal,29,18,horizontal,0.0,42560.0,46816.0,needs_review,8.313905,-49.121142,3,2,5.070157,4.795007,0.822662,strip shifts disagree
7,19__20__vertical,19,20,vertical,0.0,59584.0,63840.0,needs_review,-48.967374,-9.546770,3,2,3.321358,3.717658,0.808230,strip shifts disagree
8,30__19__horizontal,30,19,horizontal,0.0,46816.0,51072.0,needs_review,8.594737,-49.896012,3,2,4.292907,3.698911,0.830562,strip shifts disagree
9,31__20__horizontal,31,20,horizontal,0.0,51072.0,55328.0,needs_review,8.514259,-49.302330,3,2,4.889459,4.235089,0.825503,strip shifts disagree


## 3. Export registration QC images (300 dpi)

This is a separate image-registration gallery, not a segmentation gallery. It shows the original
pair composite, registered candidate composite, and each contributing FOV separately on the
registered axes. The mean blend here is deliberately diagnostic: persistent doubled structures
indicate a bad fit. The later default image policy uses one source per pixel.

Compare specific nuclei/crypt landmarks throughout the border. A smooth seam alone is not enough.
Files are under `<workspace>/<sample>/registration/visual_qc/index.html`.

In [4]:
EXPORT_REGISTRATION_QC = True
if EXPORT_REGISTRATION_QC:
    for sample in SAMPLE_KEYS:
        figures = stitch.export_registration_qc(config, sample, WORKSPACE, dpi=300, zoom_px=512)
        print('Gallery:', WORKSPACE / sample / 'registration' / 'visual_qc' / 'index.html')
        display(figures[['group', 'description', 'path']].head())

/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/code/Python/cosmx_cpsamv2_softstitch_pilot/cosmx_rgb_pilot.py:88: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  return sd.read_zarr(str(path))
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dd96d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dda210>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dd9010>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dda6f0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dd8d70>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dd9310>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dda0f0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7dbbe90>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7df8950>: 

Gallery: /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/cpsam_v2_rgb_softstitch_pilot_v1_registered_owner_v1/B20317610105_colon/registration/visual_qc/index.html


,group,description,path
0,87__88__vertical,before | candidate_pass,/host_root/nethome/reny28/Projects/CosMX_proje...
1,99__87__horizontal,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
2,88__89__vertical,before | candidate_pass,/host_root/nethome/reny28/Projects/CosMX_proje...
3,100__88__horizontal,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
4,101__89__horizontal,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...


/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/code/Python/cosmx_cpsamv2_softstitch_pilot/cosmx_rgb_pilot.py:88: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  return sd.read_zarr(str(path))
no parent found for <ome_zarr.reader.Label object at 0x7f4169b44ad0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc3f6e10>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc34c890>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc34fad0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc34c1d0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc34cdd0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7fbe330>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41e7f36b10>: None
no parent found for <ome_zarr.reader.Label object at 0x7f41fc37a750>: 

Gallery: /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/cpsam_v2_rgb_softstitch_pilot_v1_registered_owner_v1/B20971880114_colon/registration/visual_qc/index.html


,group,description,path
0,7__8__vertical,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
1,18__7__horizontal,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
2,8__9__vertical,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
3,19__8__horizontal,before | needs_review,/host_root/nethome/reny28/Projects/CosMX_proje...
4,20__9__horizontal,before | candidate_pass,/host_root/nethome/reny28/Projects/CosMX_proje...


## 4. Audit transcript observations in the registered overlap

This does **not delete anything**. It moves each source observation using the image-derived FOV
translation, preserves its frozen observation ID, and inspects mutual-nearest same-feature pairs
across different FOVs at several radii. These are **candidate correspondences**, not proven
molecule identities. The shuffled-feature comparison is descriptive, not an FDR estimate.

No same-FOV molecules are paired or collapsed. Z is not assumed co-registered across acquisitions;
its difference is recorded in the detailed candidate table. The radii are diagnostics, not deletion
thresholds.

The tables report whether a prospective spatial-ownership rule would retain exactly one, both,
or neither observation in a candidate pair. Both-kept/neither-kept cases near the ownership seam
need inspection: localization error and incomplete redetection mean spatial ownership is not
perfect molecular de-duplication.

Read `transcript_overlap_summary.csv` and `transcript_candidate_pairs.parquet`. Do not infer that
all observations excluded by ownership are duplicate molecules. Confirm that both overlapping
acquisitions actually contain transcript observations; otherwise leave ownership disabled.

In [5]:
RUN_TRANSCRIPT_AUDIT = True
if RUN_TRANSCRIPT_AUDIT:
    for sample in SAMPLE_KEYS:
        if candidates[sample]['registration_status'] != 'candidate_ready':
            print('Skipped transcript audit until registration is resolved:', sample)
            continue
        transcript_qc = stitch.audit_transcript_overlaps(
            config, sample, WORKSPACE, radii_um=(0.12, 0.25, 0.50))
        display(transcript_qc)

Skipped transcript audit until registration is resolved: B20317610105_colon
Skipped transcript audit until registration is resolved: B20971880114_colon


## Next checkpoint

Proceed to notebook 07 only after (1) the source structures align, (2) the translation diagnostics
are acceptable, and (3) duplicated acquisition observations are supported. Keep the original
pilot caches unchanged. This notebook has not edited images, labels, points, tables, or source
transformation metadata. Only the separate workspace has been written.